# SparseViT для лаборатории

Ноутбук для запуска экспериментов с SparseViT используя функции из `train_sparse_vit.py`

In [1]:
# Настройка путей (для Google Colab)
import sys
import os
from pathlib import Path

# Для Google Colab раскомментируй:
# from google.colab import drive
# drive.mount('/content/drive')
# proj_root = '/content/drive/MyDrive/nas'
# sys.path.append(proj_root)

# Для локального запуска:
# Ноутбук находится в experiments/ViT_experiments/, поэтому нужно подняться на 2 уровня вверх
proj_root = Path.cwd().parent.parent
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

print(f"Project root: {proj_root}")
print(f"Current directory: {Path.cwd()}")

Project root: /Users/user/prog/nas/self-scaled-optimized-neural-network
Current directory: /Users/user/prog/nas/self-scaled-optimized-neural-network/experiments/ViT_experiments


In [2]:
# Базовые импорты
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import time

# Импорт функций из train_sparse_vit.py
from train_sparse_vit import (
    SparseViT, DenseViT,
    train_sparse_vit, train_dense_vit,
    make_mnist_like_dataloaders, make_catsdogs_dataloaders,
    PRESETS, build_hyperparams_from_args,
    get_params_amount
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


## Инициализация Comet ML (опционально)

In [3]:
# Comet ML для логирования градиентов и метрик
experiment = None

# Раскомментируй и заполни для включения Comet ML:
from comet_ml import Experiment
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
  api_key="r0A4z5VJhRpfdOKvs2kuqouv3",
  project_name="nas_vit",
  workspace="shi-van"
)
experiment.set_name("vit_first")
print(f"Comet ML experiment started: {experiment.get_key()}")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/shi-van/nas-vit/ee18472e62b642b2a7a5999581382459



Comet ML experiment started: ee18472e62b642b2a7a5999581382459


## Настройка эксперимента

In [4]:
# Выбери датасет: "mnist", "fashion_mnist", или "catsdogs"
DATASET = "mnist"  # или "fashion_mnist", "catsdogs"
MODEL_TYPE = "sparse"  # или "dense"
PRESET_NAME = "mnist_small"  # или "fashion_small", "catsdogs"

# Пути к данным
DATA_ROOT = "./data"
PETIMAGES_ROOT = None  # Укажи путь для catsdogs: "/path/to/PetImages"
RUN_DIR = Path("./runs")

# Параметры модели (можно переопределить preset)
preset = PRESETS[PRESET_NAME]
img_size = preset.img_size
patch_size = preset.patch_size
dim = preset.dim
depth = preset.depth
num_heads = preset.num_heads
dropout = preset.dropout
batch_size = preset.batch_size
lr = preset.lr
num_epochs = preset.num_epochs

# Sparse growth/prune параметры
max_new_edges = 50
enable_expansion = (MODEL_TYPE == "sparse") and (max_new_edges > 0)

print(f"Dataset: {DATASET}")
print(f"Model: {MODEL_TYPE}")
print(f"Preset: {PRESET_NAME}")
print(f"Epochs: {num_epochs}")

Dataset: mnist
Model: sparse
Preset: mnist_small
Epochs: 100


## Создание датасетов

In [5]:
# Создание dataloaders
if DATASET in {"mnist", "fashion_mnist"}:
    dataloaders = make_mnist_like_dataloaders(
        dataset_name=DATASET,
        data_root=DATA_ROOT,
        img_size=img_size,
        batch_size=batch_size,
        num_workers=2,
        val_ratio=0.1,
        seed=42,
    )
    num_classes = 10
elif DATASET == "catsdogs":
    if PETIMAGES_ROOT is None:
        raise ValueError("PETIMAGES_ROOT должен быть указан для catsdogs")
    dataloaders = make_catsdogs_dataloaders(
        petimages_root=PETIMAGES_ROOT,
        img_size=img_size,
        batch_size=batch_size,
        num_workers=2,
        val_ratio=0.1,
        test_ratio=0.1,
        seed=42,
    )
    num_classes = 2
else:
    raise ValueError(f"Unknown dataset: {DATASET}")

print(f"Dataloaders created: {list(dataloaders.keys())}")
for phase, loader in dataloaders.items():
    print(f"  {phase}: {len(loader)} batches")

Dataloaders created: ['train', 'val', 'test']
  train: 843 batches
  val: 93 batches
  test: 157 batches


## Создание модели

In [6]:
# Создание модели
if MODEL_TYPE == "dense":
    model = DenseViT(
        num_classes=num_classes,
        img_size=img_size,
        patch_size=patch_size,
        dim=dim,
        depth=depth,
        num_heads=num_heads,
        channels=3,
        dropout=dropout,
    ).to(device)
else:
    model = SparseViT(
        num_classes=num_classes,
        img_size=img_size,
        patch_size=patch_size,
        dim=dim,
        depth=depth,
        num_heads=num_heads,
        channels=3,
        dropout=dropout,
        device=str(device),
    ).to(device)

print(f"Model params: {get_params_amount(model):,}")
print(f"Model type: {MODEL_TYPE}")

Model params: 610,826
Model type: sparse


## Настройка гиперпараметров для sparse growth/prune

In [7]:
# Гиперпараметры для sparse growth/prune (только для sparse моделей)
if MODEL_TYPE == "sparse" and enable_expansion:
    # Dataset-specific defaults
    if DATASET == "catsdogs":
        plateau_threshold = 0.02
        plateau_window_size = 2
        min_epochs_between_expansions = 6
        prune_after_epochs = 2
        expansion_thr = 0.08
        pruning_thr = 0.04
    else:
        plateau_threshold = 0.002
        plateau_window_size = 3
        min_epochs_between_expansions = 2
        prune_after_epochs = 3
        expansion_thr = 0.12
        pruning_thr = 0.06
    
    # Создаём объект args для build_hyperparams_from_args
    class Args:
        pass
    
    args = Args()
    args.max_new_edges = max_new_edges
    args.edge_score_aggregation = "mean"
    args.plateau_threshold = plateau_threshold
    args.plateau_window_size = plateau_window_size
    args.min_epochs_between_expansions = min_epochs_between_expansions
    args.prune_after_epochs = prune_after_epochs
    args.expansion_threshold = expansion_thr
    args.pruning_threshold = pruning_thr
    args.start_fully_connected = False
    
    hyperparams = build_hyperparams_from_args(depth=depth, args=args)
    
    print("Hyperparams:")
    print(f"  plateau_threshold: {plateau_threshold}")
    print(f"  plateau_window_size: {plateau_window_size}")
    print(f"  max_new_edges: {max_new_edges}")
    print(f"  expansion_thresholds: {hyperparams['expansion_thresholds']}")
    print(f"  pruning_thresholds: {hyperparams['pruning_thresholds']}")
else:
    hyperparams = None
    print("Hyperparams: None (dense model or expansion disabled)")

Hyperparams:
  plateau_threshold: 0.002
  plateau_window_size: 3
  max_new_edges: 50
  expansion_thresholds: {'qkv_layer_0': 0.12, 'qkv_layer_1': 0.12, 'qkv_layer_2': 0.12}
  pruning_thresholds: {'qkv_layer_0': 0.06, 'qkv_layer_1': 0.06, 'qkv_layer_2': 0.06}


## Логирование гиперпараметров в Comet ML

In [8]:
# Логирование гиперпараметров в Comet ML
if experiment is not None:
    params_to_log = {
        "dataset": DATASET,
        "model_type": MODEL_TYPE,
        "preset": PRESET_NAME,
        "img_size": img_size,
        "patch_size": patch_size,
        "dim": dim,
        "depth": depth,
        "num_heads": num_heads,
        "dropout": dropout,
        "batch_size": batch_size,
        "lr": lr,
        "num_epochs": num_epochs,
    }
    
    if hyperparams:
        params_to_log.update(hyperparams)
    
    experiment.log_parameters(params_to_log)
    print("Hyperparameters logged to Comet ML")

Hyperparameters logged to Comet ML


## Обучение модели

In [9]:
# Создание оптимизатора и loss функции
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=lr)

# Создание директории для результатов
run_name = f"{DATASET}_{MODEL_TYPE}_{int(time.time())}"
run_dir = RUN_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {run_dir}")

# Запуск обучения
print("Starting training...")
start_time = time.time()

if MODEL_TYPE == "dense":
    saved = train_dense_vit(
        model=model,
        dataloaders=dataloaders,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs,
        run_dir=run_dir,
        num_classes=num_classes,
        experiment=experiment,
    )
else:
    saved = train_sparse_vit(
        model=model,
        dataloaders=dataloaders,
        criterion=criterion,
        optimizer=optimizer,
        hyperparams=hyperparams,
        device=device,
        num_epochs=num_epochs,
        run_dir=run_dir,
        num_classes=num_classes,
        enable_expansion=enable_expansion,
        experiment=experiment,
    )

elapsed_time = time.time() - start_time
print(f"\nTraining completed in {elapsed_time/60:.1f} minutes")
print(f"Results saved to: {run_dir}")
print(f"  - metrics.csv: метрики каждой эпохи")
if MODEL_TYPE == "sparse":
    print(f"  - hyperparams.json: гиперпараметры sparse growth/prune")

Run directory: runs/mnist_sparse_1768923503
Starting training...
Epoch 1/100


/Users/user/miniforge3/envs/ssonn/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (7977873 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (7978025 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (7907899 > 1048576). Falling back to PNG format.


train: loss=2.328506 acc=0.1031 cm=[[22, 1, 2, 1, 8, 11, 9, 6, 0, 2], [16, 9, 2, 4, 12, 13, 5, 15, 0, 3], [30, 4, 1, 2, 9, 7, 8, 3, 0, 1], [31, 6, 1, 0, 6, 10, 6, 6, 0, 1], [27, 4, 2, 6, 9, 6, 1, 3, 0, 2], [18, 8, 2, 4, 9, 10, 7, 5, 0, 1], [18, 6, 2, 2, 5, 6, 3, 6, 0, 2], [13, 9, 3, 4, 20, 12, 5, 10, 0, 2], [28, 3, 0, 2, 8, 10, 5, 3, 0, 0], [12, 8, 2, 2, 4, 8, 6, 12, 0, 2]]


/Users/user/miniforge3/envs/ssonn/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


val: loss=2.315232 acc=0.1422 cm=[[0, 11, 0, 0, 57, 0, 0, 0, 0, 0], [0, 62, 0, 0, 9, 0, 0, 0, 0, 0], [0, 24, 0, 0, 40, 0, 0, 0, 0, 0], [0, 25, 0, 0, 35, 0, 0, 0, 0, 0], [0, 31, 0, 0, 29, 0, 0, 0, 0, 0], [0, 25, 0, 0, 35, 0, 0, 1, 0, 0], [0, 27, 0, 0, 42, 0, 0, 0, 0, 0], [0, 35, 0, 0, 37, 0, 0, 0, 0, 0], [0, 17, 0, 0, 41, 0, 0, 0, 0, 0], [0, 32, 0, 0, 25, 0, 0, 0, 0, 0]]
test: loss=2.304058 acc=0.1625 cm=[[0, 13, 0, 0, 43, 0, 0, 0, 0, 0], [0, 73, 0, 0, 2, 0, 0, 0, 0, 0], [0, 31, 0, 0, 41, 0, 0, 0, 0, 0], [0, 34, 0, 0, 31, 0, 0, 0, 0, 0], [0, 38, 0, 0, 31, 0, 0, 0, 0, 0], [0, 32, 0, 0, 27, 0, 0, 0, 0, 0], [0, 26, 0, 0, 31, 0, 0, 0, 0, 0], [0, 37, 0, 0, 24, 0, 0, 0, 0, 0], [0, 19, 0, 0, 38, 0, 0, 0, 0, 0], [0, 36, 0, 0, 33, 0, 0, 0, 0, 0]]
[plateau] waiting: val_losses=1/3
Epoch 2/100


COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (8067539 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (8036016 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (7935332 > 1048576). Falling back to PNG format.


... (7 outputs left out)

COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (8282407 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (8143268 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (7918324 > 1048576). Falling back to PNG format.


train: loss=2.080658 acc=0.2484 cm=[[38, 4, 4, 1, 1, 1, 6, 11, 0, 0], [0, 62, 0, 0, 4, 0, 2, 10, 0, 0], [37, 4, 3, 3, 0, 4, 2, 27, 0, 0], [28, 3, 6, 1, 0, 6, 0, 16, 0, 0], [14, 17, 4, 0, 5, 0, 5, 22, 0, 0], [15, 5, 1, 5, 1, 11, 0, 17, 0, 0], [24, 5, 1, 0, 1, 0, 9, 18, 1, 0], [10, 16, 1, 0, 2, 4, 2, 28, 0, 0], [27, 5, 3, 0, 3, 2, 7, 16, 2, 0], [10, 13, 1, 1, 1, 1, 4, 16, 0, 0]]
val: loss=2.038483 acc=0.2578 cm=[[44, 6, 9, 0, 4, 2, 3, 0, 0, 0], [0, 61, 0, 0, 9, 0, 1, 0, 0, 0], [26, 10, 16, 0, 1, 8, 2, 1, 0, 0], [16, 5, 21, 0, 1, 12, 5, 0, 0, 0], [13, 20, 8, 0, 16, 0, 3, 0, 0, 0], [19, 7, 16, 1, 1, 17, 0, 0, 0, 0], [25, 12, 9, 0, 8, 4, 11, 0, 0, 0], [16, 25, 10, 0, 10, 6, 5, 0, 0, 0], [26, 7, 8, 0, 7, 2, 8, 0, 0, 0], [17, 18, 4, 0, 14, 2, 2, 0, 0, 0]]


libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x1055fb010>
Traceback (most recent call last):
  File "/Users/user/miniforge3/envs/ssonn/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/Users/user/miniforge3/envs/ssonn/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1618, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/Users/user/miniforge3/envs/ssonn/lib/python3.10/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/Users/user/miniforge3/envs/ssonn/lib/python3.10/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
  File "/Users/user/miniforge3/envs/ssonn/lib/python3.10/mul

KeyboardInterrupt: 

## Завершение Comet ML эксперимента

In [ ]:
# Завершение Comet ML эксперимента
if experiment is not None:
    experiment.end()
    print("Comet ML experiment ended")
    print(f"View results at: {experiment.get_url()}")

## Визуализация результатов (опционально)

In [ ]:
# Построение графиков из metrics.csv
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = run_dir / "metrics.csv"
if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    
    # График loss
    plt.figure(figsize=(10, 4))
    for phase in ["train", "val", "test"]:
        phase_data = df[df["phase"] == phase]
        if len(phase_data) > 0:
            plt.plot(phase_data["epoch"], phase_data["loss"], label=f"{phase} loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # График accuracy
    plt.figure(figsize=(10, 4))
    for phase in ["train", "val", "test"]:
        phase_data = df[df["phase"] == phase]
        if len(phase_data) > 0:
            plt.plot(phase_data["epoch"], phase_data["acc"], label=f"{phase} acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # График количества параметров (для sparse моделей)
    if MODEL_TYPE == "sparse" and "params" in df.columns:
        plt.figure(figsize=(10, 4))
        train_data = df[df["phase"] == "train"]
        if len(train_data) > 0:
            plt.plot(train_data["epoch"], train_data["params"], label="params")
        plt.xlabel("Epoch")
        plt.ylabel("Number of Parameters")
        plt.title("Model Parameters Over Time")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
else:
    print(f"Metrics file not found: {metrics_path}")